# Snapchat Spotlight — Causal Inference, A/B Testing & Rollout Analysis

This notebook queries the user-level BigQuery analytical dataset, validates experiment integrity, estimates treatment effects, models D28 retention associations, and evaluates staged rollout decisions.

## Business Scenario

The treatment is evaluated on short-term engagement, exact-day retention, and user-experience guardrails. The final state is data-dependent.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from google.cloud import bigquery

PROJECT_ID = "driiiportfolio"
DATASET_ID = "snap_spotlight_growth"
client = bigquery.Client(project=PROJECT_ID)


In [ ]:
query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.analysis_dataset`"
df = client.query(query).to_dataframe()
print(df.shape)
df.head()


## 1. Sample Ratio Mismatch

In [ ]:
counts = df["experiment_group"].value_counts().reindex(["Control_Standard_Recs", "Variant_Algorithmic_Exploration"])
expected = np.repeat(len(df) / 2, 2)
chi2, srm_p = stats.chisquare(counts.values, f_exp=expected)
print({"control": int(counts.iloc[0]), "variant": int(counts.iloc[1]), "chi2": chi2, "p_value": srm_p})


## 2. Retention Summary

In [ ]:
retention_cols = ["retained_d1", "retained_d3", "retained_d7", "retained_d28"]
summary = df.groupby("experiment_group")[retention_cols].mean()
summary


## 3. Binary D28 Effect

In [ ]:
c = df.loc[df.experiment_group == "Control_Standard_Recs", "retained_d28"]
v = df.loc[df.experiment_group == "Variant_Algorithmic_Exploration", "retained_d28"]

diff = v.mean() - c.mean()
relative = diff / c.mean()
# Normal approximation for a two-proportion confidence interval.
p1, p2 = v.mean(), c.mean()
se = np.sqrt(p1*(1-p1)/len(v) + p2*(1-p2)/len(c))
ci_low, ci_high = diff - 1.96*se, diff + 1.96*se
z = diff / se
p_value = 2 * stats.norm.sf(abs(z))

print({
    "control_rate": c.mean(),
    "variant_rate": v.mean(),
    "absolute_difference": diff,
    "relative_difference": relative,
    "ci_low": ci_low,
    "ci_high": ci_high,
    "p_value": p_value
})


## 4. Engagement Effects — Welch t-tests

In [ ]:
for metric in ["total_shares", "avg_watch_completion", "avg_latency_ms"]:
    control = df.loc[df.experiment_group == "Control_Standard_Recs", metric]
    variant = df.loc[df.experiment_group == "Variant_Algorithmic_Exploration", metric]
    t_stat, p_val = stats.ttest_ind(variant, control, equal_var=False)
    print(metric, {"control_mean": control.mean(), "variant_mean": variant.mean(), "difference": variant.mean()-control.mean(), "t": t_stat, "p": p_val})


## 5. Logistic Regression — D28 Retention Association

This model is descriptive/predictive. Its coefficients should not be presented as causal effects of the predictors.

In [ ]:
model_df = df.copy()
model_df["variant"] = (model_df.experiment_group == "Variant_Algorithmic_Exploration").astype(int)
features = ["variant", "retained_d7", "total_shares", "avg_watch_completion", "avg_latency_ms", "negative_feedback_flag"]
X = sm.add_constant(model_df[features].astype(float))
y = model_df["retained_d28"].astype(int)
model = sm.Logit(y, X).fit(disp=False)
print(model.summary())
print("Odds ratios:")
print(np.exp(model.params))


## 6. Staged Rollout Decision Engine

Stages use progressively larger user subsets. A production implementation would use time-based exposure cohorts and live monitoring; here the stage subsets are deterministic portfolio simulation cohorts.

In [ ]:
def evaluate_stage(stage_df, exposure_rate, d28_threshold=-0.01, negative_feedback_relative_threshold=0.50):
    c = stage_df.loc[stage_df.experiment_group == "Control_Standard_Recs"]
    v = stage_df.loc[stage_df.experiment_group == "Variant_Algorithmic_Exploration"]
    d28_diff = v.retained_d28.mean() - c.retained_d28.mean()
    c_nf, v_nf = c.negative_feedback_flag.mean(), v.negative_feedback_flag.mean()
    nf_rel = (v_nf / c_nf - 1) if c_nf > 0 else np.inf
    if d28_diff < d28_threshold or nf_rel > negative_feedback_relative_threshold:
        decision = "ROLLBACK"
    else:
        decision = "ADVANCE"
    return {"exposure_rate": exposure_rate, "users": len(stage_df), "d28_diff": d28_diff, "negative_feedback_relative_change": nf_rel, "decision": decision}

stages = [0.01, 0.05, 0.10, 1.00]
stage_results = []
for i, rate in enumerate(stages, 1):
    n = max(2, int(len(df) * rate))
    stage_df = df.sort_values("user_id").head(n)
    stage_results.append({"stage": i, **evaluate_stage(stage_df, rate)})

stage_results_df = pd.DataFrame(stage_results)
stage_results_df


## 7. Final Interpretation Template

Complete the following after execution. Do not invent results.

- **Experiment integrity:** [observed SRM result]
- **Engagement:** [observed treatment effect]
- **D1/D3/D7:** [observed leading indicators]
- **D28:** [observed primary outcome, effect, CI, p-value]
- **Guardrails:** [negative feedback / latency]
- **Predictive model:** [key associations with uncertainty]
- **Rollout state:** [ADVANCE / HOLD / ROLLBACK]
- **Next action:** [evidence-linked action]